# Camera Path and Light Setup Visualization for PyBlenderRender

This notebook visualizes generated camera paths and dynamic lighting setups for PyBlenderRender, helping to fine-tune camera movements and light placements before rendering 3D models. It uses Matplotlib's 3D plotting and animation to display both the camera trajectory and light positions over time.

- Converts spherical coordinates to Cartesian for path visualization.
- Animates the camera movement along the generated path.
- Plots dynamic light positions, updating as the camera moves.
- Adjusts axis scaling to ensure both camera and lights fit in view.
- Supports interactive playback in Jupyter Notebook.

Modify the camera and lighting configurations to experiment with different setups before rendering.

## Installs and imports

In [ ]:
#!git clone https://github.com/spa-dev/PyBlenderRender.git
#%cd PyBlenderRender
#!pip install .

In [ ]:
from renderer.utils.coordinates import SphericalCoordinate
from renderer.config.camera_config import CameraConfig, SphereCoverage
from renderer.camera import camera_registry
from renderer.lighting import lighting_registry
from renderer.camera.paths import *
from renderer.lighting.base import BaseLightSetup
from renderer.lighting.setups import *
from renderer.config.lighting_config import LightingConfig, LightType, LightSetup

from IPython.display import HTML
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from typing import List, Tuple, Optional
import math
from IPython.display import HTML, display

In [1]:
# Standard library imports
import math
from typing import List, Tuple, Optional

# Third-party imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Project-specific imports
from renderer.config.camera_config import CameraConfig, SphereCoverage
from renderer.config.lighting_config import LightingConfig, LightType, LightSetup
from renderer.utils.coordinates import SphericalCoordinate
from renderer.camera import camera_registry
from renderer.camera.paths import *
from renderer.lighting import lighting_registry
from renderer.lighting.base import BaseLightSetup
from renderer.lighting.setups import *

2025-02-19 18:06:51,561 [DEBUG] PyBlenderRender: Initializing PyBlenderRender package


In [2]:
# Suppress DEBUG messages from centralized logger
import logging
from renderer.utils.logger import logger
logger.setLevel(logging.WARNING)
# Set the root logger to WARNING to suppress DEBUG messages
logging.getLogger().setLevel(logging.WARNING)

## Helper function for interactive environment

In [3]:
import sys
def is_interactive():
    """Returns True if running in Jupyter or Google Colab, else False."""
    return "ipykernel" in sys.modules or "google.colab" in sys.modules

## Main plotting function

In [4]:
class CameraLightVisualizer:
    def __init__(
        self,
        camera_positions: List[SphericalCoordinate],
        light_setup: BaseLightSetup,
        fig_size: Tuple[int, int] = (5, 5)
    ):
        self.camera_positions = camera_positions
        self.light_setup = light_setup
        
        # Initialize the plot
        self.fig = plt.figure(figsize=fig_size, dpi=80)
        self.ax = self.fig.add_subplot(111, projection='3d')
        
        # Convert camera positions to Cartesian coordinates
        self.camera_coords = [self._spherical_to_cartesian(pos) for pos in camera_positions]
        self.xs, self.ys, self.zs = zip(*self.camera_coords)
        
        # Initialize storage for light positions at each frame
        self.light_positions_frames = self._calculate_light_positions()
        
        # Initialize plot elements
        self.camera_path = None
        self.current_camera = None
        self.light_scatter = None
        
    def _spherical_to_cartesian(self, coord: SphericalCoordinate) -> Tuple[float, float, float]:
        """Convert spherical coordinates to Cartesian (Blender convention)"""
        r = coord.radius
        theta = np.radians(coord.azimuth)
        phi = np.radians(coord.elevation)
        
        x = r * np.cos(phi) * np.sin(theta)
        y = r * np.cos(phi) * np.cos(theta)
        z = r * np.sin(phi)
        
        return x, y, z
    
    def _calculate_light_positions(self) -> List[List[Tuple[float, float, float]]]:
        """Calculate light positions for each camera position"""
        light_positions_frames = []
        
        # Store original light positions
        original_positions = [(light.location.x, light.location.y, light.location.z) 
                            for light in self.light_setup._lights]
        
        # Calculate positions for each frame
        for coord in self.camera_positions:
            # Update light positions based on camera angle
            self.light_setup.update_positions(coord.azimuth)
            
            # Store current positions
            current_positions = [(light.location.x, light.location.y, light.location.z) 
                               for light in self.light_setup._lights]
            light_positions_frames.append(current_positions)
            
        # Restore original positions
        for light, pos in zip(self.light_setup._lights, original_positions):
            light.location.x, light.location.y, light.location.z = pos
            
        return light_positions_frames
  
    def _init_animation(self):
        """Initialize the animation plot elements"""
        # Plot camera path
        self.camera_path, = self.ax.plot(self.xs, self.ys, self.zs, 
                                         'b-', alpha=0.3, label='Camera Path')
    
        # Plot current camera position
        self.current_camera = self.ax.scatter([], [], [], 
                                              color='red', s=100, label='Current Camera')
    
        # Plot light positions
        self.light_scatter = self.ax.scatter([], [], [], 
                                             color='yellow', s=150, marker='*',
                                             edgecolor='orange', label='Lights')
    
        # Set labels and title
        self.ax.set_xlabel('X')
        self.ax.set_ylabel('Y')
        self.ax.set_zlabel('Z')
        
        # Update title with camera path and lighting setup
        self.ax.set_title(f'Camera Path: {camera_config.camera_path_type} | Light Setup: {lighting_config.light_setup}')
    
        # Gather all positions (camera and lights)
        all_xs = list(self.xs)
        all_ys = list(self.ys)
        all_zs = list(self.zs)
        
        for frame in self.light_positions_frames:
            light_xs, light_ys, light_zs = zip(*frame)
            all_xs.extend(light_xs)
            all_ys.extend(light_ys)
            all_zs.extend(light_zs)
    
        # Compute axis limits based on combined range
        max_range = max(
            max(all_xs) - min(all_xs),
            max(all_ys) - min(all_ys),
            max(all_zs) - min(all_zs)
        ) * 0.6  # Adjust scaling factor if needed
    
        mid_x = (max(all_xs) + min(all_xs)) * 0.5
        mid_y = (max(all_ys) + min(all_ys)) * 0.5
        mid_z = (max(all_zs) + min(all_zs)) * 0.5
    
        self.ax.set_xlim(mid_x - max_range, mid_x + max_range)
        self.ax.set_ylim(mid_y - max_range, mid_y + max_range)
        self.ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
        self.ax.legend()
        
        return self.camera_path, self.current_camera, self.light_scatter

   
    def _update_frame(self, frame):
        """Update function for animation"""
        frame = frame % len(self.camera_positions)
        
        # Update camera position
        self.current_camera._offsets3d = ([self.xs[frame]], [self.ys[frame]], [self.zs[frame]])
        
        # Update light positions
        light_pos = self.light_positions_frames[frame]
        light_xs, light_ys, light_zs = zip(*light_pos)
        self.light_scatter._offsets3d = (light_xs, light_ys, light_zs)
        
        return self.camera_path, self.current_camera, self.light_scatter
    
    def animate(self, interval: int = 50, frames: Optional[int] = None):
        """Create and display the animation"""
        if frames is None:
            frames = len(self.camera_positions)
            
        anim = FuncAnimation(
            self.fig,
            self._update_frame,
            init_func=self._init_animation,
            frames=frames,
            interval=interval,
            blit=False
        )
        
        # Handle different display environments
        if is_interactive():
            plt.close(self.fig)
            return HTML(anim.to_jshtml())
        else:
            plt.show()
            return None

def visualize_camera_light_sequence(
    camera_positions: List[SphericalCoordinate],
    light_setup: BaseLightSetup,
    interval: int = 50
) -> Optional[HTML]:
    """Convenience function to create and display the visualization"""
    visualizer = CameraLightVisualizer(camera_positions, light_setup)
    return visualizer.animate(interval=interval)

## Get lights

In [5]:
lighting_registry.available_setups

['overhead', 'random_dynamic', 'random_fixed']

In [6]:
lighting_config = LightingConfig(
    num_lights=3,
    light_type=LightType.AREA,
    light_height=18.0,
    light_radius=10.0,
    light_setup="random_dynamic",
    light_intensity=1.0
)

In [7]:
# Retrieve the appropriate light setup class from the registry based on the light setup type
LightSetupClass = lighting_registry.get_setup(lighting_config.light_setup)  

# Instantiate the retrieved light setup class, passing the lighting configuration as an argument
light_setup = LightSetupClass(lighting_config)  

# Create lights according to the chosen setup
light_setup.create_lights()  

[bpy.data.objects['Area'],
 bpy.data.objects['Area.001'],
 bpy.data.objects['Area.002']]

## Get camera

In [8]:
camera_registry.available_paths

['cube', 'orbit', 'spiral_phi', 'pole_rotation', 'spiral_lin', 'spiral_phased']

In [9]:
camera_config = CameraConfig(
    distance=10.0,  # Adjust based on your scene size
    camera_path_type="spiral_lin",  # Replace with your path type
    #camera_density=60, # Number of camera positions for orbit and spiral_phi
    angular_step=30 # Base angular step for spiral_lin and spiral_phased
)

generator = camera_registry.get_generator(camera_config.camera_path_type)
camera_positions = generator.generate_positions(camera_config)

## Action!

In [10]:
# After generating camera positions and setting up lights
visualize_camera_light_sequence(
    camera_positions=camera_positions,
    light_setup=light_setup,
    interval=100  # milliseconds between frames
)